# `nnsightful` Cheat Sheet

`nnsightful` is a higher-level library built on top of [nnsight](https://nnsight.net/) that packages common mechanistic interpretability workflows into simple function calls with interactive visualizations. This cheat sheet covers the functions available in `nnsightful` and how they relate to the underlying `nnsight` operations.

**Contents:**
1. [Setup](#1-setup)
2. [Logit Lens](#2-logit-lens) — Inspect intermediate layer predictions
3. [Layer-Level Activation Patching](#3-layer-level-activation-patching) — Identify critical layers via causal mediation
4. [Attention Head Splitting](#4-attention-head-splitting) — Decompose attention output by individual head (for head-level patching)
5. [Visualization](#5-visualization) — Display functions for notebooks

## 1. Setup

In [1]:
try:
    import google.colab  # type: ignore
    !pip install git+https://github.com/AdamBelfki3/nnsightful.git
    from IPython.display import clear_output
    clear_output()
except Exception:
    pass

In [ ]:
import os
from nnterp import StandardizedTransformer
from nnsightful import logit_lens, activation_patching
import torch

REMOTE = True

# Load NDIF key
os.environ["NDIF_API_KEY"] = "YOUR API KEY"

/Users/gsarti/Documents/projects/ndif-ecosystem/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_NAME = "meta-llama/Llama-3.1-8B"

# remote=False + allow_dispatch=False avoids registering nnterp for
# pickle-by-value (which pulls in non-whitelisted internal modules).
# check_renaming=False skips scan-based validation (incompatible with transformers 5.x).
model = StandardizedTransformer(
    MODEL_NAME,
    device_map="auto",
    remote=False,
    allow_dispatch=not REMOTE,
    check_renaming=not REMOTE,
)

# Useful model constants (available via nnterp's standardized accessors)
N_HEADS = model.num_heads        # 32
H_DIM = model.config.head_dim   # 128
MODEL_DIM = model.hidden_size    # 4096
LAYERS = model.num_layers        # 32

---

## 2. Logit Lens

Projects each layer's hidden state through the model's unembedding head to see what the model "would predict" at every intermediate layer.

### `logit_lens(model, prompt, remote=False)`

| Parameter | Type | Description |
|-----------|------|-------------|
| `model` | `StandardizedTransformer` | The loaded nnterp model |
| `prompt` | `str` | Input text to analyze |
| `remote` | `bool` | Run on the NDIF remote server (default `False`) |

**Returns** a `LogitLensData` with fields: `.layers`, `.input` (tokens as strings), `.tracked` (per-position token probability trajectories), `.topk` (per-layer top-k tokens), `.entropy`.

Call `.display()` on the result to render an interactive heatmap.

**Under the hood (nnsight):** For each layer, this reads `model.layers_output[i]`, applies `model.project_on_vocab()`, and collects the resulting logits.

---

## 3. Residual Stream Activation Patching

Tests whether copying a hidden state from a **source prompt** into a **target prompt** at a specific layer changes the model's prediction. Sweeps across all layers to identify which ones are critical.

### `activation_patching(model, src_prompt, tgt_prompt, src_pos, tgt_pos, tgt_freeze, remote=False)`

| Parameter | Type | Description |
|-----------|------|-------------|
| `model` | `StandardizedTransformer` | The loaded nnterp model |
| `src_prompt` | `str` | Prompt whose activations are copied |
| `tgt_prompt` | `str` | Prompt that receives the patched activations |
| `src_pos` | `List[int \| List[int]]` | Source token position(s). Use `int` for a single position, or `[start, end]` to average a range |
| `tgt_pos` | `List[int]` | Target token position(s) to patch into (same length as `src_pos`) |
| `tgt_freeze` | `List[int]` | Target position(s) to freeze at clean values in subsequent layers, avoiding the second-order effect from activation patching. |
| `remote` | `bool` | Run on the NDIF remote server (default `False`) |

**Returns** an `ActivationPatchingData` with fields: `.lines` (probabilities per token per layer), `.ranks`, `.prob_diffs` (vs. clean run), `.tokenLabels`.

Call `.display()` to render an interactive line chart with Probability / Prob Delta / Rank views. Pass `tokens=[0, 1, ...]` to select which tracked tokens to show.

**Under the hood (nnsight):** Uses `model.session()` with three traces: (1) read source activations from `model.layers_output[i]`, (2) collect clean target activations, (3) per-layer patching runs where it overwrites `hs[0, pos]` and freezes other positions in later layers.

### Finding token positions

Use the tokenizer to figure out which positions to patch:

```python
for i, t in enumerate(model.tokenizer.encode("your prompt")):
    print(f"  {i}: {repr(model.tokenizer.decode([t]))}")
```

Typically, patch the **last token** of the relevant content (e.g., the final `:` in an ICL prompt).

In [4]:
# Example: Inspect token positions for your prompts
src_prompt = "amount: candidad, win: gañar, dreams:"
tgt_prompt = "dog: chien, run: courir, flowers:"

print("Source prompt tokens:")
for i, t in enumerate(model.tokenizer.encode(src_prompt)):
    print(f"  {i}: {repr(model.tokenizer.decode([t]))}")

print("\nTarget prompt tokens:")
for i, t in enumerate(model.tokenizer.encode(tgt_prompt)):
    print(f"  {i}: {repr(model.tokenizer.decode([t]))}")

Source prompt tokens:
  0: '<|begin_of_text|>'
  1: 'amount'
  2: ':'
  3: ' candid'
  4: 'ad'
  5: ','
  6: ' win'
  7: ':'
  8: ' ga'
  9: 'ñ'
  10: 'ar'
  11: ','
  12: ' dreams'
  13: ':'

Target prompt tokens:
  0: '<|begin_of_text|>'
  1: 'dog'
  2: ':'
  3: ' ch'
  4: 'ien'
  5: ','
  6: ' run'
  7: ':'
  8: ' cour'
  9: 'ir'
  10: ','
  11: ' flowers'
  12: ':'


---

## 4. Attention Head Splitting

`nnsightful` operates at the **layer level**. For **attention head-level** patching (Part 2 of the workshop), you need to decompose a layer's attention output into individual head contributions. This requires working directly with `nnsight`.

The attention output projection (`o_proj`) takes the concatenated head outputs and projects them back to model dimension:

$$\text{AttnOut} = \text{concat}(h_1, h_2, \ldots, h_H) \cdot W_O$$

We can split this to get each head's individual contribution to the residual stream:

### `split_heads_W_O(o_inp, W_O, n_heads, h_dim, model_dim)`

| Parameter | Type | Description |
|-----------|------|-------------|
| `o_inp` | `Tensor [B, T, N_HEADS * H_DIM]` | Input to the output projection (pre-`o_proj` activations) |
| `W_O` | `Tensor [MODEL_DIM, N_HEADS * H_DIM]` | The `o_proj` weight matrix |
| `n_heads` | `int` | Number of attention heads |
| `h_dim` | `int` | Dimension per head |
| `model_dim` | `int` | Model hidden dimension |

**Returns** a tensor of shape `[B, T, N_HEADS, MODEL_DIM]` — each head's contribution to the attention output.

The sum across heads reconstructs the original output: `o_out_by_head.sum(dim=2) ≈ o_proj.output`.

**Accessing these values inside a trace (nnsight):**
```python
with model.trace(prompt, remote=REMOTE):
    layer = model.layers[LAYER_IDX]
    o_inp = layer.self_attn.o_proj.input     # [B, T, N_HEADS * H_DIM]
    W_O = layer.self_attn.o_proj.weight       # [MODEL_DIM, N_HEADS * H_DIM]
    o_out_by_head = split_heads_W_O(o_inp, W_O, N_HEADS, H_DIM, MODEL_DIM)
```

**Note:** With `StandardizedTransformer` from nnterp, layers are accessed via `model.layers[i]` (standardized) rather than the architecture-specific `model.model.layers[i]`.

**Overriding the output projection with modified heads:**

After modifying individual heads in `o_out_by_head`, you can skip the original `o_proj` and inject the modified result:

```python
layer.self_attn.o_proj.skip(o_out_by_head.sum(dim=2))
```

`skip(value)` replaces the module's output with the given value, bypassing the original computation.

In [5]:
def split_heads_W_O(o_inp, W_O, n_heads, h_dim, model_dim):
    """Split attention output projection into individual head contributions.

    Args:
        o_inp:     Input to o_proj, shape [B, T, n_heads * h_dim]
        W_O:       o_proj weight matrix, shape [model_dim, n_heads * h_dim]
        n_heads:   Number of attention heads
        h_dim:     Dimension per head
        model_dim: Model hidden dimension

    Returns:
        Tensor of shape [B, T, n_heads, model_dim] — per-head contributions.
    """
    b, t, _ = tuple(o_inp.shape)
    o_inp_by_head = o_inp.view(b, t, n_heads, h_dim)       # [B, T, H, h_dim]
    W_O_by_head = W_O.view(model_dim, n_heads, h_dim)      # [D, H, h_dim]
    o_out_by_head = torch.einsum(
        "bthk,dhk->bthd",
        o_inp_by_head, W_O_by_head,
    )                                                        # [B, T, H, D]
    return o_out_by_head

In [6]:
# Example: Verify head splitting reconstructs the original output
PROMPT = "The Eiffel Tower is located in the city of"
LAYER = 5

with model.session(remote=REMOTE):
    # Get the full attention output
    with model.trace(PROMPT):
        o_out = model.layers[LAYER].self_attn.o_proj.output.save()

    # Get per-head contributions and verify they sum to the full output
    with model.trace(PROMPT):
        o_inp = model.layers[LAYER].self_attn.o_proj.input
        o_out_by_head = split_heads_W_O(
            o_inp,
            model.layers[LAYER].self_attn.o_proj.weight,
            N_HEADS, H_DIM, MODEL_DIM,
        ).save()

print(f"Full attention output:   {o_out.shape}")       # [1, T, 4096]
print(f"Per-head contributions:  {o_out_by_head.shape}") # [1, T, 32, 4096]
print(f"Reconstruction matches:  {torch.allclose(o_out, o_out_by_head.sum(dim=2), atol=1e-2)}")

⬇ Downloading: 100%|██████████| 2.54M/2.54M [00:01<00:00]


Full attention output:   torch.Size([1, 12, 4096])
Per-head contributions:  torch.Size([1, 12, 32, 4096])
Reconstruction matches:  True


---

## 5. Visualization

`nnsightful` provides interactive JavaScript widgets that render directly in Jupyter notebooks. These are the same components used in the Workbench web application.

### `display_logit_lens(data, ui_state=None, width="80%", height="450px", dark_mode=False)`

Renders a logit lens heatmap with trajectory chart. Usually called via `LogitLensData.display()`.

### `display_activation_patching(data, options=None, width="80%", height="400px")`

Renders a line chart for activation patching results with mode-switching buttons (Probability, Prob Delta, Rank). Usually called via `ActivationPatchingData.display()`.

| Parameter | Type | Description |
|-----------|------|-------------|
| `data` | `dict` or Pydantic model | The data to visualize (auto-converted from Pydantic models) |
| `width` | `str` | CSS width of the container (default `"80%"`) |
| `height` | `str` | CSS height of the container |
| `dark_mode` | `bool` | Use dark mode (logit lens only) |

### `display_line_plot(data, options=None, width="80%", height="300px")`

A generic line plot widget. `data` should have `lines` (list of lists) and optionally `labels` (list of strings).

In [7]:
# Visualization functions can also be imported directly
from nnsightful.viz import display_line_plot

# Example: display a custom line plot
display_line_plot(
    data={"lines": [[0.1, 0.3, 0.7, 0.9], [0.8, 0.5, 0.2, 0.1]], "labels": ["token A", "token B"]},
    options={"title": "Example line plot"},
)

---

## Quick Reference

| What you want to do | `nnsightful` function | Underlying `nnsight` pattern |
|---|---|---|
| See what each layer predicts | `logit_lens(model, prompt)` | Read `model.layers_output[i]`, apply `model.project_on_vocab()` |
| Find which layers matter for a behavior | `activation_patching(model, src, tgt, ...)` | `model.session()` with multiple `model.trace()`: save source activations, patch into target, read logits |
| Decompose attention by head | `split_heads_W_O(...)` (provided above) | Read `o_proj.input` + `o_proj.weight`, reshape and einsum |